# Exploratory Data Analysis — German Credit Dataset

**FinSecure AI** | Credit Risk Assessment Platform

This notebook explores the *Statlog (German Credit Data)* dataset fetched from OpenML (`credit-g`, version 1). The target variable is **class**: `good` (low risk) vs `bad` (high risk).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from model.utils import load_dataset

plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)

## 1. Dataset Overview

In [ ]:
df = load_dataset()
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
df.head()

## 2. Feature Descriptions

| Feature | Description |
|---------|-------------|
| checking_status | Status of existing checking account |
| duration | Duration of credit in months |
| credit_history | Credit history category |
| purpose | Purpose of the loan |
| credit_amount | Credit amount in DM |
| savings_status | Status of savings account/bonds |
| employment | Present employment duration |
| installment_commitment | Installment rate as % of disposable income |
| personal_status | Personal status and sex |
| other_parties | Other debtors or guarantors |
| residence_since | Years at present residence |
| property_magnitude | Most valuable property |
| age | Age in years |
| other_payment_plans | Other installment plans |
| housing | Housing situation |
| existing_credits | Number of existing credits at this bank |
| job | Job type |
| num_dependents | Number of dependents |
| own_telephone | Whether applicant owns a telephone |
| foreign_worker | Whether applicant is a foreign worker |
| **class** | **Target: good or bad credit** |

In [ ]:
df.info()

## 3. Target Distribution & Class Imbalance

In [ ]:
target_counts = df['class'].value_counts()
target_pct = df['class'].value_counts(normalize=True) * 100

print('Target distribution:')
print(target_counts)
print('\nPercentages:')
print(target_pct.round(2))

imbalance_ratio = target_counts.max() / target_counts.min()
print(f'\nClass imbalance ratio (majority/minority): {imbalance_ratio:.2f}:1')

fig, ax = plt.subplots(figsize=(6, 4))
target_counts.plot(kind='bar', ax=ax, color=['#22c55e', '#ef4444'])
ax.set_title('Target Class Distribution')
ax.set_xlabel('Class')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 4. Missing Values

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_df[missing_df['missing_count'] > 0] if missing.sum() > 0 else print('No missing values detected.')

## 5. Summary Statistics (Numerical Features)

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
if 'class' in numeric_cols:
    numeric_cols.remove('class')
df[numeric_cols].describe().T

## 6. Categorical Feature Analysis

In [ ]:
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols = [c for c in cat_cols if c != 'class']

for col in cat_cols:
    print(f'\n--- {col} ({df[col].nunique()} unique) ---')
    print(df[col].value_counts().head(10))

In [ ]:
fig, axes = plt.subplots(4, 4, figsize=(16, 14))
axes = axes.flatten()

for i, col in enumerate(cat_cols[:16]):
    df[col].value_counts().head(8).plot(kind='bar', ax=axes[i], color='#3b82f6')
    axes[i].set_title(col, fontsize=9)
    axes[i].tick_params(axis='x', rotation=45, labelsize=7)

for j in range(len(cat_cols[:16]), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Categorical Feature Distributions', y=1.02)
plt.tight_layout()
plt.show()

## 7. Numerical Feature Analysis

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(df[col], bins=30, color='#6366f1', edgecolor='white')
    axes[i].set_title(col)

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Feature Distributions', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    df.boxplot(column=col, by='class', ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_xlabel('class')

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Numerical Features by Target Class', y=1.02)
plt.tight_layout()
plt.show()

## 8. Correlation Heatmap

In [ ]:
df_corr = df.copy()
df_corr['class_encoded'] = (df_corr['class'] == 'bad').astype(int)

corr = df_corr[numeric_cols + ['class_encoded']].corr()

fig, ax = plt.subplots(figsize=(10, 8))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title('Correlation Heatmap (Numerical Features + Target)')
plt.tight_layout()
plt.show()

## 9. Important Insights

Key observations from this EDA:

1. **Dataset size**: ~1,000 applicants with 20 features plus the target — a classic tabular credit scoring benchmark.
2. **Class imbalance**: The dataset is imbalanced (~70% good, ~30% bad). Stratified splitting and balanced metrics (F1, ROC-AUC) are important.
3. **Missing values**: The German Credit dataset typically has no missing values; imputation is still included in the pipeline for robustness.
4. **Mixed feature types**: 13 categorical and 7 numerical features require separate preprocessing strategies (encoding vs scaling).
5. **Duration & credit amount**: Longer loan durations and higher credit amounts tend to correlate with higher default risk.
6. **Checking/savings status**: Account status features are strong predictors — applicants with little or no checking/savings balance show higher risk.
7. **Age & employment**: Younger applicants and those with shorter employment history may carry elevated risk.
8. **Non-linear relationships**: Box plots reveal non-linear patterns between features and the target, motivating tree-based models like XGBoost over purely linear approaches.